# Notebook 04: Full End-to-End Fine-Tuning

In this notebook, we perform true end-to-end training. **All weights of the Moirai encoder and the classification heads are updated**.


In [4]:
import torch
import pandas as pd
from IPython.display import display

from heads import (
    MeanPoolingClassifier,
    SingleScaleAttentionClassifier,
    SingleScaleMultiHeadClassifier,
    HierarchicalMultiHeadClassifier,
)
from models import FullMaskOnlyWrapper, FullHeadWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [5*1e-5]
wd_grid = [0.01, 0.05]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from utils import set_seed
set_seed(42)
print("Seed 42 — experiments are reproducible.")

Seed 42 — experiments are reproducible.


In [6]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [7]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 1. Baseline: Linear Model on Mask Embedding Only (Full FT)

In [8]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullMaskOnlyWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mask_only.loc["Mask Only", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mask Only"] = metrics

LR=5e-05| WD=0.01


1.9557560740447626
1.6724274410464899
1.5020063902304424
1.397839158531127
1.3534049687346792
1.2937771682816792
1.2678925080028007
1.2569621985520774
1.2420395796861106
1.2532262850583085
1.2223279098185098
1.2354646514101726
1.276772572742245
1.2705284037241122
1.277150048473017
1.3410246197770281
 Early stopping : 15
Val Loss: 1.2223
LR=5e-05| WD=0.05
1.923425577520355
1.6905082890657874
1.5212360727108591
1.416074535711025
1.3438226895603707
1.289501541029147
1.2668763457275019
1.2690761254085758
1.2555510619791543
1.252039344330144
1.2369243400852854
1.2455453882372476
1.2628619069975566
1.2656653169693985
1.2729210417445114
1.2732638430789234
 Early stopping : 15
Val Loss: 1.2369
LR=5e-05| WD=0.01
1.9833258865325432
1.703437368075053
1.5352505600549342
1.4189283450444539
1.3415294895327188
1.3018071767760486
1.2889735621165455
1.2838413977041476
1.2710853437098062
1.2903875190068066
1.3072209435749829
1.3466109270002784
1.4167458012821228
1.4168044474066757
 Early stopping : 13
V

In [9]:
print("Mask Only - Accuracy")
display(df_mask_only.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mask Only - Accuracy


Patch Size,8,16,32,64
Mask Only,0.6172,0.6042,0.6127,0.6148



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4982,0.3511,0.3603,0.5867,0.6172,0.5682


## 2. Baseline: Mean Pooling

In [10]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mean_pool.loc["Mean Pooling", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mean Pooling"] = metrics

LR=5e-05| WD=0.01
1.8771864272714631
1.6595724414034587
1.4909877534804306
1.370907456894231
1.2913283477953779
1.2314876695958579
1.211476974371003
1.159789517643006
1.1221631736290165
1.1024565192742077
1.0910511986026918
1.0821167492285007
1.0943050869112092
1.1005079629944592
1.1106700577386996
1.1310124465120517
1.1902915423478537
 Early stopping : 16
Val Loss: 1.0821
LR=5e-05| WD=0.05
1.9292028396110223
1.61991264761948
1.4430696566899617
1.340724391665885
1.2822382062431272
1.2346756603659652
1.1910737655996306
1.202671972716727
1.1357636286960384
1.1259828195339296
1.1360633819083856
1.124668512887102
1.1814975622223645
1.1483718253732698
1.1550882134011122
1.1783677010032219
1.209700835429556
 Early stopping : 16
Val Loss: 1.1247
LR=5e-05| WD=0.01
2.1007137453652978
1.8027606553178492
1.598025477998625
1.4486483035048818
1.329054060021067
1.258341961759862
1.20005109833508
1.1736103344738968
1.1579992790532305
1.1534899473190308
1.1481977623652637
1.1469894812359074
1.16058454

In [11]:
print("Mean Pooling - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mean Pooling - Accuracy


Patch Size,8,16,32,64
Mean Pooling,0.6456,0.6375,0.6127,0.635



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4982,0.3511,0.3603,0.5867,0.6172,0.5682
Mean Pooling,0.6456,0.4804,0.3776,0.3842,0.5968,0.6456,0.6006


## 3. Advanced Pooling: Attention & MHA (Full FT)

In [12]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 64

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, metrics_attn = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[("Attention (Base)", mode), patch] = metrics_attn["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics_attn

        # 2. Multi-Head Attention
        _, metrics_mha = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(f"MHA (Heads={NUM_HEADS})", mode), patch] = metrics_mha["Accuracy"]

print("Attention & MHA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.1249
LR=5e-05| WD=0.05
 Early stopping : 17
Val Loss: 1.1389
LR=5e-05| WD=0.01
 Early stopping : 16
Val Loss: 1.1072
LR=5e-05| WD=0.05
 Early stopping : 16
Val Loss: 1.0940
LR=5e-05| WD=0.01
 Early stopping : 17
Val Loss: 1.1312
LR=5e-05| WD=0.05
 Early stopping : 16
Val Loss: 1.1301
LR=5e-05| WD=0.01
 Early stopping : 14
Val Loss: 1.1138
LR=5e-05| WD=0.05
 Early stopping : 14
Val Loss: 1.0927
LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.2008
LR=5e-05| WD=0.05
 Early stopping : 12
Val Loss: 1.1777
LR=5e-05| WD=0.01
 Early stopping : 14
Val Loss: 1.1015
LR=5e-05| WD=0.05
 Early stopping : 13
Val Loss: 1.1541
LR=5e-05| WD=0.01
 Early stopping : 13
Val Loss: 1.1913
LR=5e-05| WD=0.05
 Early stopping : 14
Val Loss: 1.2453
LR=5e-05| WD=0.01
 Early stopping : 12
Val Loss: 1.1906
LR=5e-05| WD=0.05
 Early stopping : 15
Val Loss: 1.1446
LR=5e-05| WD=0.01
 Early stopping : 14
Val Loss: 1.2103
LR=5e-05| WD=0.05
 Early stopping : 14
Val Loss:

Patch Size                                8       16      32      64
Model            Mode                                               
Attention (Base) shared_context       0.6342  0.6265  0.6115  0.6379
                 independent_context  0.6436  0.6200  0.6221  0.6237
MHA (Heads=64)   shared_context       0.6419  0.6265  0.6022  0.6212
                 independent_context  0.6452  0.6180  0.6196  0.6200


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4982,0.3511,0.3603,0.5867,0.6172,0.5682
Mean Pooling,0.6456,0.4804,0.3776,0.3842,0.5968,0.6456,0.6006
Attention (shared_context),0.6342,0.4110,0.3668,0.3657,0.5673,0.6342,0.5802
Attention (independent_context),0.6436,0.4878,0.3712,0.3767,0.5938,0.6436,0.5886


## 4. Ablation Study: Number of Attention Heads

In [ ]:
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384]

df_heads_ablation8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation8.index.name = "Mode"
df_heads_ablation8.columns.name = "Num Heads (Patch 8)"
df_heads_ablation16 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation16.index.name = "Mode"
df_heads_ablation16.columns.name = "Num Heads (Patch 16)"

for PATCH in [8]:
    for mode in MODES:
        for heads in HEADS_TO_TEST:
            _, metrics = universal_grid_search(
                model_class=FullHeadWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                    "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
                },
                train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
                wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
            )
            if PATCH == 8:
                df_heads_ablation8.loc[mode, heads] = metrics["Accuracy"]
                df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics
            else:
                df_heads_ablation16.loc[mode, heads] = metrics["Accuracy"]

LR=5e-05| WD=0.01
2.12088098370932
1.845229807907973
1.611679833109786
1.4406581341735716
1.323679369639575
1.2591128872662056
1.1866150357858922
1.2201556665141409
1.1637994322350356
1.1130645226656906
1.105252109892
1.0764240423838298
1.1461550336543138
1.110879964944793
1.1327197590494544
1.2263570452124122
1.307886018016474
 Early stopping : 16
Val Loss: 1.0764
LR=5e-05| WD=0.05
2.1203194614348373
1.9178879968519134
1.7125776288955192
1.5719773381706175
1.4035615009990166
1.3156019623686628
1.2389395963854906
1.2363874728117532
1.202962916071822
1.1805909067634646
1.191416535920244
1.1307379831143511
1.16977503823071
1.2549357830993528
1.2986834824569826
1.328151975220781
1.4507200165492733
 Early stopping : 16
Val Loss: 1.1307
LR=5e-05| WD=0.01
2.1162129727805534
1.85354199351334
1.6266274403750411
1.4618353843688965
1.3613802785795879
1.2996986435680855
1.2525347306476375
1.208372115119686
1.195849155022846
1.1832443369113332
1.1439820663715765
1.1893991512980888
1.16211183284356

In [ ]:
print("Ablation: Num Heads (Patch = 8) - Accuracy")
display(df_heads_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Ablation: Num Heads (Patch = 8) - Accuracy


Num Heads (Patch 8),4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6476,0.6261,0.6233,0.6395,0.6346,0.6330,0.6310
independent_context,0.6436,0.6387,0.6395,0.6395,0.6350,0.6334,0.6354



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4982,0.3511,0.3603,0.5867,0.6172,0.5682
Mean Pooling,0.6456,0.4804,0.3776,0.3842,0.5968,0.6456,0.6006
Attention (shared_context),0.6342,0.4110,0.3668,0.3657,0.5673,0.6342,0.5802
Attention (independent_context),0.6436,0.4878,0.3712,0.3767,0.5938,0.6436,0.5886
MHA-4 (shared_context),0.6476,0.4558,0.3743,0.3727,0.5947,0.6476,0.5944
MHA-8 (shared_context),0.6261,0.4090,0.3550,0.3463,0.5611,0.6261,0.5571
MHA-16 (shared_context),0.6233,0.4337,0.3881,0.3990,0.5837,0.6233,0.5965
MHA-32 (shared_context),0.6395,0.4045,0.3716,0.3714,0.5691,0.6395,0.5828
MHA-64 (shared_context),0.6346,0.4716,0.3876,0.3840,0.5838,0.6346,0.5907


## 5. Hierarchical MHA (Full FT)

In [ ]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode})"] = metrics_hier

LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.2653
LR=5e-05| WD=0.05
 Early stopping : 19
Val Loss: 1.2004
LR=5e-05| WD=0.01
 Early stopping : 17
Val Loss: 1.2204
LR=5e-05| WD=0.05
 Early stopping : 17
Val Loss: 1.2681
LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.3088
LR=5e-05| WD=0.05
 Early stopping : 17
Val Loss: 1.2394
LR=5e-05| WD=0.01
 Early stopping : 18
Val Loss: 1.2928
LR=5e-05| WD=0.05
 Early stopping : 17
Val Loss: 1.2959
LR=5e-05| WD=0.01
 Early stopping : 17
Val Loss: 1.2641
LR=5e-05| WD=0.05
 Early stopping : 14
Val Loss: 1.2692
LR=5e-05| WD=0.01
 Early stopping : 14
Val Loss: 1.2758
LR=5e-05| WD=0.05
 Early stopping : 15
Val Loss: 1.2755
LR=5e-05| WD=0.01
 Early stopping : 16
Val Loss: 1.3162
LR=5e-05| WD=0.05
 Early stopping : 14
Val Loss: 1.2880
LR=5e-05| WD=0.01
 Early stopping : 12
Val Loss: 1.3121
LR=5e-05| WD=0.05
 Early stopping : 18
Val Loss: 1.2711


In [ ]:
print("Hierarchical MHA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Hierarchical MHA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5949,0.5787,0.5738,0.5876
independent_context,0.5961,0.5827,0.5726,0.5977



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4982,0.3511,0.3603,0.5867,0.6172,0.5682
Mean Pooling,0.6456,0.4804,0.3776,0.3842,0.5968,0.6456,0.6006
Attention (shared_context),0.6342,0.4110,0.3668,0.3657,0.5673,0.6342,0.5802
Attention (independent_context),0.6436,0.4878,0.3712,0.3767,0.5938,0.6436,0.5886
MHA-4 (shared_context),0.6476,0.4558,0.3743,0.3727,0.5947,0.6476,0.5944
MHA-8 (shared_context),0.6261,0.4090,0.3550,0.3463,0.5611,0.6261,0.5571
MHA-16 (shared_context),0.6233,0.4337,0.3881,0.3990,0.5837,0.6233,0.5965
MHA-32 (shared_context),0.6395,0.4045,0.3716,0.3714,0.5691,0.6395,0.5828
MHA-64 (shared_context),0.6346,0.4716,0.3876,0.3840,0.5838,0.6346,0.5907


In [ ]:
# ── Export Results to CSV ─────────────────────────────────────────────────────
import os
import numpy as np

os.makedirs("results_csv", exist_ok=True)

FINETUNING = "04_FullFT"
MHA_HEADS  = 64
MODE_LABEL = {'shared_context': 'shared', 'independent_context': 'indep'}
rows = []

for patch in PATCH_SIZES:
    # Mean Pooling (no mode)
    acc = float(df_mean_pool.loc['Mean Pooling', patch])
    if patch == 8:
        rec = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro Recall'])
        f1  = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro F1'])
    else:
        rec, f1 = float('nan'), float('nan')
    rows.append({'finetuning': FINETUNING, 'pooling': 'Mean', 'patch_size': patch,
                 'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Attention — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_adv_single.loc[('Attention (Base)', mode), patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Attention-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # MHA-64 — both modes (only evaluated at patch=8 in the ablation)
    for mode, lbl in MODE_LABEL.items():
        if patch == 8:
            acc = float(df_heads_ablation8.loc[mode, MHA_HEADS])
            rec = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro F1'])
        else:
            acc = float(df_adv_single.loc[('MHA (Heads=64)', mode), patch])
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'MHA-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Hierarchical — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_hierarchical.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Hierarchical ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Hierarchical ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Hierarchical-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

df_nb04 = pd.DataFrame(rows)
df_nb04.to_csv("results_csv/nb04_results.csv", index=False)
print("Saved results_csv/nb04_results.csv")
display(df_nb04)

Saved results_csv/nb04_results.csv


,finetuning,pooling,patch_size,accuracy,macro_recall,macro_f1
0,04_FullFT,Mean,8,0.645580,0.377568,0.384207
1,04_FullFT,Attention-shared,8,0.634225,0.366776,0.365705
2,04_FullFT,Attention-indep,8,0.643552,0.371154,0.376716
3,04_FullFT,MHA-shared,8,0.634631,0.387556,0.383954
4,04_FullFT,MHA-indep,8,0.635036,0.365740,0.361883
5,04_FullFT,Hierarchical-shared,8,0.594891,0.327132,0.314330
6,04_FullFT,Hierarchical-indep,8,0.596107,0.317319,0.305447
7,04_FullFT,Mean,16,0.637470,NaN,NaN
8,04_FullFT,Attention-shared,16,0.626521,NaN,NaN
9,04_FullFT,Attention-indep,16,0.620032,NaN,NaN


In [ ]:
df_patch_8_metrics.to_csv("results_csv/nb04_patch_8_metrics.csv", index=True)